# Target Analysis for CyberGuard AI

This notebook defines and justifies the supervised target for the ML component. It does not train the final LightGBM model.

Key objective:
- estimate how important a vulnerability is for a given enterprise context
- maintain a defensible, leakage-safe label
- enable retrospective backtesting where feasible

In [ ]:
# Setup cell: install missing dependencies and configure the project root for Colab.
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()

if "google.colab" in sys.modules:
    repo_url = os.getenv("GITHUB_REPO_URL")
    if repo_url:
        print(f"Cloning repository from {repo_url}")
        !git clone {repo_url} /content/cyberguard-ai
        %cd /content/cyberguard-ai
        REPO_ROOT = Path.cwd()
    else:
        print("Set GITHUB_REPO_URL before running this notebook in Colab.")
        print("Example: export GITHUB_REPO_URL=https://github.com/your-org/your-repo.git")

if not str(REPO_ROOT).endswith("CyberGuard-Ai"):
    candidate_dirs = [Path("."), Path("/content/cyberguard-ai"), Path("/workspace"), Path("/content")]
    for path in candidate_dirs:
        if (path / "data" / "processed" / "risk_features.csv").exists():
            REPO_ROOT = path
            break

os.chdir(REPO_ROOT)
print(f"Project root: {REPO_ROOT}")

# Install only missing dependencies if needed.
# This is intentionally light and avoids local Windows-specific paths.
!python -m pip install -q pandas numpy scikit-learn matplotlib plotly lightgbm

# Import reusable project utilities.
from src.ml.target_definition import load_risk_features, summarize_candidate_targets, choose_target

DATA_PATH = REPO_ROOT / "data" / "processed" / "risk_features.csv"
df = load_risk_features(DATA_PATH)
print(f"Loaded rows: {len(df)}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Inspect the dataset and confirm the label candidates are possible.
df.head()

In [ ]:
# Review all target candidates in a structured way.
candidates = summarize_candidate_targets()
for item in candidates:
    print("\n===", item["candidate"], "===")
    for key in ["meaning", "data_needed", "leakage_risk", "backtesting", "fit"]:
        print(f"- {key}: {item[key]}")

In [ ]:
# Final target recommendation with explicit logic.
selected = choose_target()
for key, value in selected.items():
    print(f"{key}: {value}")

## Recommended final target

The selected target is:

Future KEV appearance within a future time window.

This is the most defensible operational target because it measures whether a vulnerability becomes a confirmed exploited issue after the observation date. That aligns with the project objective: identify vulnerabilities that are likely to become urgent and should be prioritized under limited remediation hours.

### Why this target is preferred
- It is future-oriented rather than contemporaneous.
- It aligns with exploit-driven risk prioritization.
- It supports time-based backtesting and evaluation.
- It is more actionable than a purely static snapshot label.

### Guardrails
- Never train on features that include a future KEV flag for the same vulnerability.
- Always define the label using a later observation window than the feature window.
- Use time-based splits and do not shuffle by CVE_ID.
- Do not declare the model final until this target is approved and documented.

In [ ]:
# Save the target definition report to the repository reports folder.
from src.ml.target_definition import write_report

report_path = write_report(REPO_ROOT / "reports" / "target_definition.md")
print(f"Target definition report saved to: {report_path}")